# Eksperymenty E1-E5: Shapley vs Owen vs Banzhaf

Ten notatnik zawiera eksperymenty porównujące metody Shapley, Owen i Banzhaf.


In [7]:
# Importy
import sys
import os
sys.path.insert(0, os.path.join(os.getcwd(), "Functions"))

from Functions.utils.imports import *
from Functions.data_loading import load_event_log, discover_tree_inductive, assign_tau_labels, LOG_PATHS, NOISE_LEVELS
from Functions.process_tree import enumerate_tau, count_nodes
from Functions.tree_conversion import tree_to_named_pattern_expression
from Functions.logical_spec import WorkflowPatternTemplate
from Functions.properties import extract_ini_fin, build_full_spec, evaluate_property
from Functions.shapley import shapley_mc_permutations, owen_value_permutations, banzhaf_random_coalitions
from Functions.ranking import sort_players, jaccard_at_k, kendall_tau_rank, method_stats, compare_methods, aggregate_group_values, coverage_ratio
from Functions.players import list_players_from_expression, derive_player_blocks, derive_subtree_groups, _enumerate_pattern_nodes
from Functions.coalition import build_coalition_artifacts
from Functions.io import load_cached_shapley_results, split_player_id, save_json, append_rows_csv, ensure_dir
from Functions.utils.constants import PATTERN_RULES_PATH, OUT_ROOT

# Załaduj dane - sprawdź czy są w globals(), jeśli nie, załaduj z cache/odtwórz
if "PATTERNS" not in globals() or "TEMPLATES" not in globals():
    print("PATTERNS/TEMPLATES nie są dostępne - odtwarzanie...")
    
    # Ładowanie logów zdarzeń
    EVENT_LOGS = {}
    for name, path in LOG_PATHS.items():
        EVENT_LOGS[name] = load_event_log(path)
    
    # Odkrywanie drzew procesu
    TREES = {}
    for log_name, log_obj in EVENT_LOGS.items():
        for noise in NOISE_LEVELS:
            tree = discover_tree_inductive(log_obj, noise=noise)
            tree = assign_tau_labels(tree)
            TREES[(log_name, noise)] = tree
    
    # Konwersja drzew na nazwane wyrażenia wzorców
    PATTERNS = {}
    for key, tree in TREES.items():
        named_expr = tree_to_named_pattern_expression(tree)
        PATTERNS[key] = named_expr
    
    # Ładowanie szablonów wzorców
    TEMPLATES = WorkflowPatternTemplate.load_pattern_property_set(PATTERN_RULES_PATH)
    
    print(f"✓ Odtworzono PATTERNS ({len(PATTERNS)} konfiguracji) i TEMPLATES")
else:
    print("✓ PATTERNS i TEMPLATES są już dostępne")

# Załaduj SHAPLEY_RESULTS z cache
if "SHAPLEY_RESULTS" not in globals():
    SHAPLEY_RESULTS = load_cached_shapley_results()
    if SHAPLEY_RESULTS:
        print(f"✓ Załadowano SHAPLEY_RESULTS z cache ({len(SHAPLEY_RESULTS)} konfiguracji)")
    else:
        print("⚠ SHAPLEY_RESULTS nie są dostępne w cache. Uruchom Shapley_mining.ipynb aby je wygenerować.")
else:
    print("✓ SHAPLEY_RESULTS są już dostępne")


✓ PATTERNS i TEMPLATES są już dostępne
✓ SHAPLEY_RESULTS są już dostępne


## Eksperyment E1 — Shapley vs Owen vs Banzhaf


In [ ]:
def _fetch_shapley_values(log_name: str, noise: float, property_type: str):
    """Pobierz wartości Shapley z SHAPLEY_RESULTS."""
    for row in SHAPLEY_RESULTS:
        if row["log"] == log_name and row["noise"] == noise and row["property"] == property_type:
            return row["phi_mc"], row["mc_meta"].get("seconds", 0.0)
    raise ValueError(f"No Shapley results for {(log_name, noise, property_type)}")

# E1: Wszystkie konfiguracje - wszystkie logi, wszystkie poziomy szumu, wszystkie właściwości
E1_CONFIGS = [
    {"log": log_name, "noise": noise, "property": prop}
    for log_name in LOG_PATHS.keys()
    for noise in NOISE_LEVELS
    for prop in ["satisfiability", "liveness", "safety"]
]
E1_PARAMS = {"owen_perms": 600, "banzhaf_samples": 1500, "top_k": 10}

E1_METHOD_STATS = []
E1_COMPARISONS = []
E1_RESULTS = {}
J_KEY = f"jaccard@{E1_PARAMS['top_k']}"

total_configs = len(E1_CONFIGS)
print(f"[E1] Uruchamianie eksperymentu na {total_configs} konfiguracjach...")

for idx, cfg in enumerate(E1_CONFIGS, 1):
    log_name = cfg["log"]
    noise = cfg["noise"]
    prop = cfg["property"]
    
    try:
        named_expr = PATTERNS[(log_name, noise)]
        block_map = derive_player_blocks(named_expr)

        print(f"\n[E1] [{idx}/{total_configs}] log={log_name}, noise={noise}, property={prop} -> start")

        shapley_vals, shapley_runtime = _fetch_shapley_values(log_name, noise, prop)
        owen_vals, owen_meta = owen_value_permutations(
            named_expr, TEMPLATES, prop,
            block_partition=block_map,
            n_perm=E1_PARAMS["owen_perms"], seed=904 + idx, progress_every=50
        )
        banzhaf_bundle, banzhaf_meta = banzhaf_random_coalitions(
            named_expr, TEMPLATES, prop,
            n_samples=E1_PARAMS["banzhaf_samples"], seed=2718 + idx, progress_every=150
        )
        banzhaf_vals = banzhaf_bundle["raw"]

        E1_RESULTS[(log_name, noise, prop)] = {
            "Shapley_MC": shapley_vals,
            "Owen": owen_vals,
            "Banzhaf": banzhaf_vals
        }

        stats_rows = [
            method_stats("Shapley_MC", shapley_vals, shapley_runtime),
            method_stats("Owen", owen_vals, owen_meta["seconds"]),
            method_stats("Banzhaf", banzhaf_vals, banzhaf_meta["seconds"])
        ]
        for row in stats_rows:
            row.update({"log": log_name, "noise": noise, "property": prop})
            E1_METHOD_STATS.append(row)

        comparisons = [
            ("Shapley vs Owen", shapley_vals, owen_vals),
            ("Shapley vs Banzhaf", shapley_vals, banzhaf_vals),
            ("Owen vs Banzhaf", owen_vals, banzhaf_vals)
        ]
        for label, vals_a, vals_b in comparisons:
            row = compare_methods(label, vals_a, vals_b, k=E1_PARAMS["top_k"])
            row.update({"log": log_name, "noise": noise, "property": prop})
            E1_COMPARISONS.append(row)
            
        print(f"[E1] [{idx}/{total_configs}] ✓ zakończono")
    except Exception as e:
        print(f"[E1] [{idx}/{total_configs}] ✗ błąd: {e}")
        continue

E1_STATS_DF = pd.DataFrame(E1_METHOD_STATS)
E1_COMP_DF = pd.DataFrame(E1_COMPARISONS)

print(f"\n[E1] Zakończono eksperyment. Wyniki dla {len(E1_METHOD_STATS)} kombinacji metody+konfiguracja.")
print("\n[E1] Method-level stats:")
display(E1_STATS_DF)
print("[E1] Ranking agreement metrics:")
display(E1_COMP_DF)


[E1] Uruchamianie eksperymentu na 36 konfiguracjach...

[E1] [1/36] log=running_example, noise=0.0, property=satisfiability -> start
      ... Owen progress 50/600 (blocks=5) [0.8s]
      ... Owen progress 100/600 (blocks=5) [0.8s]
      ... Owen progress 150/600 (blocks=5) [0.8s]
      ... Owen progress 200/600 (blocks=5) [0.8s]
      ... Owen progress 250/600 (blocks=5) [0.8s]
      ... Owen progress 300/600 (blocks=5) [0.8s]
      ... Owen progress 350/600 (blocks=5) [0.8s]
      ... Owen progress 400/600 (blocks=5) [0.8s]
      ... Owen progress 450/600 (blocks=5) [0.8s]
      ... Owen progress 500/600 (blocks=5) [0.8s]
      ... Owen progress 550/600 (blocks=5) [0.8s]
      ... Owen progress 600/600 (blocks=5) [0.8s]
      ... Banzhaf progress 150/1500 (players=6) [0.0s]
      ... Banzhaf progress 300/1500 (players=6) [0.0s]
      ... Banzhaf progress 450/1500 (players=6) [0.0s]
      ... Banzhaf progress 600/1500 (players=6) [0.0s]
      ... Banzhaf progress 750/1500 (players=6) 

,method,std,runtime_s,max,min,log,noise,property
0,Shapley_MC,0.216922,0.291802,0.267000,-0.442000,running_example,0.0,satisfiability
1,Owen,0.215129,0.799458,0.283333,-0.431667,running_example,0.0,satisfiability
2,Banzhaf,0.226200,0.019032,0.379814,-0.373856,running_example,0.0,satisfiability
3,Shapley_MC,0.000000,0.224421,0.000000,0.000000,running_example,0.0,liveness
4,Owen,0.000000,0.180344,0.000000,0.000000,running_example,0.0,liveness
...,...,...,...,...,...,...,...,...
103,Owen,0.000000,1.058925,0.000000,0.000000,bpi_2012,1.0,liveness
104,Banzhaf,0.000000,0.185505,0.000000,0.000000,bpi_2012,1.0,liveness
105,Shapley_MC,0.000000,1.318506,0.000000,0.000000,bpi_2012,1.0,safety
106,Owen,0.000000,1.013030,0.000000,0.000000,bpi_2012,1.0,safety


[E1] Ranking agreement metrics:


,pair,kendall_tau,jaccard@10,log,noise,property
0,Shapley vs Owen,1.000000,1.0,running_example,0.0,satisfiability
1,Shapley vs Banzhaf,0.733333,1.0,running_example,0.0,satisfiability
2,Owen vs Banzhaf,0.733333,1.0,running_example,0.0,satisfiability
3,Shapley vs Owen,1.000000,1.0,running_example,0.0,liveness
4,Shapley vs Banzhaf,1.000000,1.0,running_example,0.0,liveness
...,...,...,...,...,...,...
103,Shapley vs Banzhaf,1.000000,1.0,bpi_2012,1.0,liveness
104,Owen vs Banzhaf,1.000000,1.0,bpi_2012,1.0,liveness
105,Shapley vs Owen,1.000000,1.0,bpi_2012,1.0,safety
106,Shapley vs Banzhaf,1.000000,1.0,bpi_2012,1.0,safety


In [9]:
E1_OUT_DIR = os.path.join(OUT_ROOT, "experiments", "E1")
ensure_dir(E1_OUT_DIR)
E1_STATS_DF.to_csv(os.path.join(E1_OUT_DIR, "E1_method_stats.csv"), index=False)
E1_COMP_DF.to_csv(os.path.join(E1_OUT_DIR, "E1_comparisons.csv"), index=False)
E1_RESULTS_JSON = {f"{k[0]}:{k[1]}:{k[2]}": v for k, v in E1_RESULTS.items()}
save_json(os.path.join(E1_OUT_DIR, "E1_results.json"), E1_RESULTS_JSON)
print(f"\n[E1] ✓ Zapisano wyniki do {E1_OUT_DIR}/")


[E1] ✓ Zapisano wyniki do ../Docs/Problems/shapley_values/experiments/E1/


## Eksperyment E2 — Group influence (Owen vs ΣShapley)


In [ ]:
# E2: Wszystkie konfiguracje - wszystkie logi, wszystkie poziomy szumu, wszystkie właściwości
E2_CONFIGS = [
    {"log": log_name, "noise": noise, "property": prop}
    for log_name in LOG_PATHS.keys()
    for noise in NOISE_LEVELS
    for prop in ["satisfiability", "liveness", "safety"]
]
E2_PARAMS = {"owen_perms": 600, "top_k": 3}

E2_GROUP_TABLES = []
E2_GROUP_COVERAGE = []

total_configs = len(E2_CONFIGS)
print(f"[E2] Uruchamianie eksperymentu na {total_configs} konfiguracjach...")

for idx, cfg in enumerate(E2_CONFIGS, 1):
    log_name = cfg["log"]
    noise = cfg["noise"]
    prop = cfg["property"]
    
    try:
        base_key = (log_name, noise)
        named_expr = PATTERNS[base_key]
        subtree_groups = derive_subtree_groups(named_expr, drop_root=True)
        
        if not subtree_groups:
            print(f"[E2] [{idx}/{total_configs}] log={log_name}, noise={noise}, property={prop} -> brak grup (pomijanie)")
            continue

        nodes_lookup = {node["id"]: node for node in _enumerate_pattern_nodes(named_expr)}
        block_partition = derive_player_blocks(named_expr, drop_root=False)

        print(f"\n[E2] [{idx}/{total_configs}] log={log_name}, noise={noise}, property={prop} -> start")
        shapley_vals, _ = _fetch_shapley_values(log_name, noise, prop)
        owen_vals, owen_meta = owen_value_permutations(
            named_expr, TEMPLATES, prop,
            block_partition=block_partition,
            n_perm=E2_PARAMS["owen_perms"],
            seed=1500 + idx,
            progress_every=50
        )

        shapley_group_vals = aggregate_group_values(shapley_vals, subtree_groups)
        owen_group_vals = aggregate_group_values(owen_vals, subtree_groups)

        rows = []
        for group_id, members in subtree_groups.items():
            node_meta = nodes_lookup.get(group_id, {})
            rows.append({
                "group_id": group_id,
                "pattern": f"{node_meta.get('name', '?')}@{node_meta.get('label', '?')}",
                "size": len(members),
                "shapley_sum": shapley_group_vals.get(group_id, 0.0),
                "owen_sum": owen_group_vals.get(group_id, 0.0)
            })

        df = pd.DataFrame(rows)
        df["abs_shapley"] = df["shapley_sum"].abs()
        df["abs_owen"] = df["owen_sum"].abs()
        df["delta"] = df["owen_sum"] - df["shapley_sum"]
        df_sorted = df.sort_values("abs_owen", ascending=False).reset_index(drop=True)
        E2_GROUP_TABLES.append(df_sorted.assign(log=log_name, noise=noise, property=prop))

        cov_shapley = coverage_ratio(shapley_group_vals, E2_PARAMS["top_k"])
        cov_owen = coverage_ratio(owen_group_vals, E2_PARAMS["top_k"])
        cmp_stats = compare_methods("Group-level", shapley_group_vals, owen_group_vals, k=E2_PARAMS["top_k"])

        coverage_row = {
            "log": log_name,
            "noise": noise,
            "property": prop,
            "coverage_shapley": cov_shapley,
            "coverage_owen": cov_owen,
            "kendall_tau": cmp_stats["kendall_tau"],
            f"J@{E2_PARAMS['top_k']}": cmp_stats[f"jaccard@{E2_PARAMS['top_k']}"]
        }
        E2_GROUP_COVERAGE.append(coverage_row)
        
        print(f"[E2] [{idx}/{total_configs}] ✓ zakończono")
    except Exception as e:
        print(f"[E2] [{idx}/{total_configs}] ✗ błąd: {e}")
        continue

E2_GROUP_SUMMARY_DF = pd.concat(E2_GROUP_TABLES, ignore_index=True) if E2_GROUP_TABLES else pd.DataFrame()
E2_COVERAGE_DF = pd.DataFrame(E2_GROUP_COVERAGE)

print(f"\n[E2] Zakończono eksperyment. Wyniki dla {len(E2_GROUP_COVERAGE)} konfiguracji.")
print("\n[E2] Coverage & agreement summary:")
display(E2_COVERAGE_DF)
print("Experiment E2 completed.")

[E2] Uruchamianie eksperymentu na 36 konfiguracjach...

[E2] [1/36] log=running_example, noise=0.0, property=satisfiability -> start
      ... Owen progress 50/600 (blocks=5) [0.0s]
      ... Owen progress 100/600 (blocks=5) [0.0s]
      ... Owen progress 150/600 (blocks=5) [0.0s]
      ... Owen progress 200/600 (blocks=5) [0.0s]
      ... Owen progress 250/600 (blocks=5) [0.0s]
      ... Owen progress 300/600 (blocks=5) [0.0s]
      ... Owen progress 350/600 (blocks=5) [0.0s]
      ... Owen progress 400/600 (blocks=5) [0.0s]
      ... Owen progress 450/600 (blocks=5) [0.0s]
      ... Owen progress 500/600 (blocks=5) [0.0s]
      ... Owen progress 550/600 (blocks=5) [0.0s]
      ... Owen progress 600/600 (blocks=5) [0.0s]
[E2] [1/36] ✓ zakończono

[E2] [2/36] log=running_example, noise=0.0, property=liveness -> start
      ... Owen progress 50/600 (blocks=5) [0.0s]
      ... Owen progress 100/600 (blocks=5) [0.0s]
      ... Owen progress 150/600 (blocks=5) [0.0s]
      ... Owen progres

,log,noise,property,coverage_shapley,coverage_owen,kendall_tau,J@3
0,running_example,0.00,satisfiability,1.000000,1.000000,1.000000,1.0
1,running_example,0.00,liveness,0.000000,0.000000,1.000000,1.0
2,running_example,0.00,safety,0.000000,0.000000,1.000000,1.0
3,running_example,0.25,satisfiability,1.000000,1.000000,1.000000,1.0
4,running_example,0.25,liveness,0.000000,0.000000,1.000000,1.0
5,running_example,0.25,safety,0.000000,0.000000,1.000000,1.0
6,running_example,0.50,satisfiability,1.000000,1.000000,1.000000,1.0
7,running_example,0.50,liveness,0.000000,0.000000,1.000000,1.0
8,running_example,0.50,safety,0.000000,0.000000,1.000000,1.0
9,running_example,1.00,satisfiability,1.000000,1.000000,1.000000,1.0


Experiment E2 completed.


In [10]:
E2_OUT_DIR = os.path.join(OUT_ROOT, "experiments", "E2")
ensure_dir(E2_OUT_DIR)
E2_COVERAGE_DF.to_csv(os.path.join(E2_OUT_DIR, "E2_coverage.csv"), index=False)
if not E2_GROUP_SUMMARY_DF.empty:
    E2_GROUP_SUMMARY_DF.to_csv(os.path.join(E2_OUT_DIR, "E2_group_summary.csv"), index=False)
print(f"[E2] ✓ Zapisano wyniki do {E2_OUT_DIR}/")

[E2] ✓ Zapisano wyniki do ../Docs/Problems/shapley_values/experiments/E2/


## Eksperyment E3 — Noise robustness (Shapley vs Owen vs Banzhaf)


In [ ]:
from Functions.ranking import mean_std_across_values

# E3: Wszystkie logi, wszystkie właściwości, wszystkie poziomy szumu
E3_CONFIG = {
    "logs": list(LOG_PATHS.keys()),
    "properties": ["satisfiability", "liveness", "safety"],
    "noise_levels": NOISE_LEVELS,
    "top_k": 10,
    "owen_perms": 400,
    "banzhaf_samples": 1200,
    "seeds": [511, 611, 711]
}

E3_METHODS = ["Shapley_MC", "Owen", "Banzhaf"]
E3_METHOD_DATA = {
    log_name: {
        m: {prop: {} for prop in E3_CONFIG["properties"]} 
        for m in E3_METHODS
    }
    for log_name in E3_CONFIG["logs"]
}

J_KEY_E3 = f"jaccard@{E3_CONFIG['top_k']}"

def _multi_seed_estimate(factory, seeds):
    per_seed_values = []
    runtimes = []
    for seed in seeds:
        vals, meta = factory(seed)
        per_seed_values.append(vals)
        runtimes.append(meta.get("seconds", 0.0))
    avg_vals, _, avg_std = mean_std_across_values(per_seed_values)
    runtime = float(np.mean(runtimes)) if runtimes else 0.0
    return avg_vals, avg_std, runtime

total_configs = len(E3_CONFIG["logs"]) * len(E3_CONFIG["noise_levels"]) * len(E3_CONFIG["properties"])
config_count = 0

for log_name in E3_CONFIG["logs"]:
    for noise in E3_CONFIG["noise_levels"]:
        try:
            named_expr = PATTERNS[(log_name, noise)]
            block_partition = derive_player_blocks(named_expr, drop_root=False)
            config_count += 1
            print(f"\n[E3] [{config_count}/{total_configs}] log={log_name}, noise={noise} -> building method estimates")
            
            for prop in E3_CONFIG["properties"]:
                shapley_vals, shapley_runtime = _fetch_shapley_values(log_name, noise, prop)
                E3_METHOD_DATA[log_name]["Shapley_MC"][prop][noise] = {
                    "values": shapley_vals,
                    "avg_std": 0.0,
                    "runtime_s": shapley_runtime
                }

                def _owen_factory(seed):
                    vals, meta = owen_value_permutations(
                        named_expr, TEMPLATES, prop,
                        block_partition=block_partition,
                        n_perm=E3_CONFIG["owen_perms"],
                        seed=seed,
                        progress_every=max(25, E3_CONFIG["owen_perms"] // 8)
                    )
                    return vals, meta

                def _banzhaf_factory(seed):
                    bundle, meta = banzhaf_random_coalitions(
                        named_expr, TEMPLATES, prop,
                        n_samples=E3_CONFIG["banzhaf_samples"],
                        seed=seed,
                        progress_every=max(50, E3_CONFIG["banzhaf_samples"] // 10)
                    )
                    return bundle["raw"], meta

                owen_avg, owen_std, owen_runtime = _multi_seed_estimate(_owen_factory, E3_CONFIG["seeds"])
                banz_avg, banz_std, banz_runtime = _multi_seed_estimate(_banzhaf_factory, E3_CONFIG["seeds"])

                E3_METHOD_DATA[log_name]["Owen"][prop][noise] = {
                    "values": owen_avg,
                    "avg_std": owen_std,
                    "runtime_s": owen_runtime
                }
                E3_METHOD_DATA[log_name]["Banzhaf"][prop][noise] = {
                    "values": banz_avg,
                    "avg_std": banz_std,
                    "runtime_s": banz_runtime
                }
        except Exception as e:
            print(f"[E3] [{config_count}/{total_configs}] ✗ błąd dla log={log_name}, noise={noise}: {e}")
            continue

E3_STABILITY_ROWS = []
for log_name in E3_CONFIG["logs"]:
    for method in E3_METHODS:
        for prop in E3_CONFIG["properties"]:
            if 0.0 not in E3_METHOD_DATA[log_name][method][prop]:
                continue
            baseline_vals = E3_METHOD_DATA[log_name][method][prop][0.0]["values"]
            for noise in E3_CONFIG["noise_levels"]:
                if noise not in E3_METHOD_DATA[log_name][method][prop]:
                    continue
                current = E3_METHOD_DATA[log_name][method][prop][noise]
                if noise == 0.0:
                    tau = 1.0
                    jacc = 1.0
                else:
                    stats = compare_methods(
                        f"{method} noise {noise}",
                        current["values"],
                        baseline_vals,
                        k=E3_CONFIG["top_k"]
                    )
                    tau = stats.get("kendall_tau", 0.0)
                    jacc = stats.get(J_KEY_E3, 0.0)
                E3_STABILITY_ROWS.append({
                    "log": log_name,
                    "method": method,
                    "property": prop,
                    "noise": noise,
                    "kendall_tau_vs0": tau,
                    f"J@{E3_CONFIG['top_k']}_vs0": jacc,
                    "avg_seed_std": current.get("avg_std", 0.0),
                    "runtime_s": current.get("runtime_s", 0.0)
                })

E3_STABILITY_DF = pd.DataFrame(E3_STABILITY_ROWS)
print(f"\n[E3] Zakończono eksperyment. Wyniki dla {len(E3_STABILITY_ROWS)} kombinacji.")
print("\n[E3] Ranking stability vs baseline noise=0.0:")
display(E3_STABILITY_DF)
print("Experiment E3 completed.")



[E3] [1/36] log=running_example, noise=0.0 -> building method estimates
      ... Owen progress 50/400 (blocks=5) [0.0s]
      ... Owen progress 100/400 (blocks=5) [0.0s]
      ... Owen progress 150/400 (blocks=5) [0.0s]
      ... Owen progress 200/400 (blocks=5) [0.0s]
      ... Owen progress 250/400 (blocks=5) [0.0s]
      ... Owen progress 300/400 (blocks=5) [0.0s]
      ... Owen progress 350/400 (blocks=5) [0.0s]
      ... Owen progress 400/400 (blocks=5) [0.0s]
      ... Owen progress 50/400 (blocks=5) [0.0s]
      ... Owen progress 100/400 (blocks=5) [0.0s]
      ... Owen progress 150/400 (blocks=5) [0.0s]
      ... Owen progress 200/400 (blocks=5) [0.0s]
      ... Owen progress 250/400 (blocks=5) [0.0s]
      ... Owen progress 300/400 (blocks=5) [0.0s]
      ... Owen progress 350/400 (blocks=5) [0.0s]
      ... Owen progress 400/400 (blocks=5) [0.0s]
      ... Owen progress 50/400 (blocks=5) [0.0s]
      ... Owen progress 100/400 (blocks=5) [0.0s]
      ... Owen progress 150/40

,log,method,property,noise,kendall_tau_vs0,J@10_vs0,avg_seed_std,runtime_s
0,running_example,Shapley_MC,satisfiability,0.00,1.000000,1.000000,0.0,0.291802
1,running_example,Shapley_MC,satisfiability,0.25,1.000000,1.000000,0.0,0.046021
2,running_example,Shapley_MC,satisfiability,0.50,1.000000,1.000000,0.0,0.033695
3,running_example,Shapley_MC,satisfiability,1.00,1.000000,1.000000,0.0,0.030094
4,running_example,Shapley_MC,liveness,0.00,1.000000,1.000000,0.0,0.224421
...,...,...,...,...,...,...,...,...
103,bpi_2012,Banzhaf,liveness,1.00,0.000000,0.000000,0.0,0.152258
104,bpi_2012,Banzhaf,safety,0.00,1.000000,1.000000,0.0,30.414578
105,bpi_2012,Banzhaf,safety,0.25,0.783333,0.176471,0.0,19.708552
106,bpi_2012,Banzhaf,safety,0.50,0.820513,0.176471,0.0,12.614415


Experiment E3 completed.


In [11]:
E3_OUT_DIR = os.path.join(OUT_ROOT, "experiments", "E3")
ensure_dir(E3_OUT_DIR)
E3_STABILITY_DF.to_csv(os.path.join(E3_OUT_DIR, "E3_stability.csv"), index=False)
save_json(os.path.join(E3_OUT_DIR, "E3_method_data.json"), E3_METHOD_DATA)
print(f"[E3] ✓ Zapisano wyniki do {E3_OUT_DIR}/")

[E3] ✓ Zapisano wyniki do ../Docs/Problems/shapley_values/experiments/E3/


## Eksperyment E4 — Pruning via Shapley/Owen/Banzhaf


In [ ]:
import time

# E4: Wszystkie konfiguracje - wszystkie logi, wszystkie poziomy szumu, wszystkie właściwości
E4_CONFIGS = [
    {"log": log_name, "noise": noise, "property": prop}
    for log_name in LOG_PATHS.keys()
    for noise in NOISE_LEVELS
    for prop in ["satisfiability", "liveness", "safety"]
]
E4_PARAMS = {
    "thresholds": [0.01, 0.03, 0.05],
    "owen_perms": 500,
    "banzhaf_samples": 1500
}

E4_PRUNING_ROWS = []

total_configs = len(E4_CONFIGS)
print(f"[E4] Uruchamianie eksperymentu na {total_configs} konfiguracjach...")

for idx, cfg in enumerate(E4_CONFIGS, 1):
    log_name = cfg["log"]
    noise = cfg["noise"]
    prop = cfg["property"]
    
    try:
        baseline_key = (log_name, noise)
        named_expr = PATTERNS[baseline_key]
        players_total = len(list_players_from_expression(named_expr))
        block_partition = derive_player_blocks(named_expr, drop_root=False)

        method_values = {}
        method_meta = {}

        print(f"\n[E4] [{idx}/{total_configs}] log={log_name}, noise={noise}, property={prop} -> start")

        shapley_vals, shapley_runtime = _fetch_shapley_values(log_name, noise, prop)
        method_values["Shapley_MC"] = shapley_vals
        method_meta["Shapley_MC"] = {"runtime_s": shapley_runtime}

        o_vals, o_meta = owen_value_permutations(
            named_expr, TEMPLATES, prop,
            block_partition=block_partition,
            n_perm=E4_PARAMS["owen_perms"],
            seed=1818 + idx,
            progress_every=max(25, E4_PARAMS["owen_perms"] // 8)
        )
        method_values["Owen"] = o_vals
        method_meta["Owen"] = {"runtime_s": o_meta.get("seconds", 0.0)}

        b_bundle, b_meta = banzhaf_random_coalitions(
            named_expr, TEMPLATES, prop,
            n_samples=E4_PARAMS["banzhaf_samples"],
            seed=4242 + idx,
            progress_every=max(50, E4_PARAMS["banzhaf_samples"] // 10)
        )
        method_values["Banzhaf"] = b_bundle["raw"]
        method_meta["Banzhaf"] = {"runtime_s": b_meta.get("seconds", 0.0)}

        for method, values in method_values.items():
            for theta in E4_PARAMS["thresholds"]:
                keep_ids = {pid for pid, val in values.items() if abs(val) >= theta}
                masked_expr, spec_text, ini, fin = build_coalition_artifacts(named_expr, keep_ids, TEMPLATES)
                eval_start = time.time()
                sat_ok = bool(evaluate_property(spec_text, "satisfiability"))
                liv_ok = bool(evaluate_property(spec_text, "liveness", ini, fin))
                saf_ok = bool(evaluate_property(spec_text, "safety", ini, fin))
                eval_seconds = time.time() - eval_start

                E4_PRUNING_ROWS.append({
                    "log": log_name,
                    "noise": noise,
                    "property": prop,
                    "method": method,
                    "threshold": theta,
                    "players_total": players_total,
                    "players_kept": len(keep_ids),
                    "kept_ratio": len(keep_ids) / players_total if players_total else 0.0,
                    "sat_ok": sat_ok,
                    "liv_ok": liv_ok,
                    "saf_ok": saf_ok,
                    "prep_runtime_s": method_meta.get(method, {}).get("runtime_s", 0.0),
                    "eval_runtime_s": eval_seconds
                })
        
        print(f"[E4] [{idx}/{total_configs}] ✓ zakończono")
    except Exception as e:
        print(f"[E4] [{idx}/{total_configs}] ✗ błąd: {e}")
        continue

E4_PRUNING_DF = pd.DataFrame(E4_PRUNING_ROWS)
print(f"\n[E4] Zakończono eksperyment. Wyniki dla {len(E4_PRUNING_ROWS)} kombinacji.")
print("[E4] Pruning outcomes:")
display(E4_PRUNING_DF)
print("Experiment E4 completed.")


[E4] Uruchamianie eksperymentu na 36 konfiguracjach...

[E4] [1/36] log=running_example, noise=0.0, property=satisfiability -> start
      ... Owen progress 62/500 (blocks=5) [0.0s]
      ... Owen progress 124/500 (blocks=5) [0.0s]
      ... Owen progress 186/500 (blocks=5) [0.0s]
      ... Owen progress 248/500 (blocks=5) [0.0s]
      ... Owen progress 310/500 (blocks=5) [0.0s]
      ... Owen progress 372/500 (blocks=5) [0.0s]
      ... Owen progress 434/500 (blocks=5) [0.0s]
      ... Owen progress 496/500 (blocks=5) [0.0s]
      ... Banzhaf progress 150/1500 (players=6) [0.0s]
      ... Banzhaf progress 300/1500 (players=6) [0.0s]
      ... Banzhaf progress 450/1500 (players=6) [0.0s]
      ... Banzhaf progress 600/1500 (players=6) [0.0s]
      ... Banzhaf progress 750/1500 (players=6) [0.0s]
      ... Banzhaf progress 900/1500 (players=6) [0.0s]
      ... Banzhaf progress 1050/1500 (players=6) [0.0s]
      ... Banzhaf progress 1200/1500 (players=6) [0.0s]
      ... Banzhaf progress

,log,noise,property,method,threshold,players_total,players_kept,kept_ratio,sat_ok,liv_ok,saf_ok,prep_runtime_s,eval_runtime_s
0,running_example,0.0,satisfiability,Shapley_MC,0.01,6,4,0.666667,True,True,False,0.291802,0.000017
1,running_example,0.0,satisfiability,Shapley_MC,0.03,6,4,0.666667,True,True,False,0.291802,0.000010
2,running_example,0.0,satisfiability,Shapley_MC,0.05,6,4,0.666667,True,True,False,0.291802,0.000009
3,running_example,0.0,satisfiability,Owen,0.01,6,4,0.666667,True,True,False,0.018260,0.000010
4,running_example,0.0,satisfiability,Owen,0.03,6,4,0.666667,True,True,False,0.018260,0.000009
...,...,...,...,...,...,...,...,...,...,...,...,...,...
319,bpi_2012,1.0,safety,Owen,0.03,8,0,0.000000,True,True,False,0.134083,0.000009
320,bpi_2012,1.0,safety,Owen,0.05,8,0,0.000000,True,True,False,0.134083,0.000009
321,bpi_2012,1.0,safety,Banzhaf,0.01,8,0,0.000000,True,True,False,0.141060,0.000009
322,bpi_2012,1.0,safety,Banzhaf,0.03,8,0,0.000000,True,True,False,0.141060,0.000009


Experiment E4 completed.


In [12]:
E4_OUT_DIR = os.path.join(OUT_ROOT, "experiments", "E4")
ensure_dir(E4_OUT_DIR)
E4_PRUNING_DF.to_csv(os.path.join(E4_OUT_DIR, "E4_pruning.csv"), index=False)
print(f"[E4] ✓ Zapisano wyniki do {E4_OUT_DIR}/")

[E4] ✓ Zapisano wyniki do ../Docs/Problems/shapley_values/experiments/E4/


## Eksperyment E5 — Mini policy / kiedy której metody używać


In [ ]:
required_vars = ["E1_METHOD_STATS", "E1_COMP_DF", "E2_COVERAGE_DF", "E3_STABILITY_DF", "E4_PRUNING_DF"]

for var in required_vars:
    assert var in globals(), f"Missing data from earlier experiments: {var}"

group_coverage = {
    "Shapley_MC": float(E2_COVERAGE_DF["coverage_shapley"].mean()) if not E2_COVERAGE_DF.empty else None,
    "Owen": float(E2_COVERAGE_DF["coverage_owen"].mean()) if not E2_COVERAGE_DF.empty else None
}

E5_POLICY_ROWS = []
for method in E3_METHODS:
    stability_rows = E3_STABILITY_DF[E3_STABILITY_DF["method"] == method]
    pruning_mid = E4_PRUNING_DF[(E4_PRUNING_DF["method"] == method) & (E4_PRUNING_DF["threshold"] == 0.03)]
    
    avg_stability = stability_rows["kendall_tau_vs0"].mean() if not stability_rows.empty else 0.0
    avg_pruning_kept = pruning_mid["kept_ratio"].mean() if not pruning_mid.empty else 0.0
    
    E5_POLICY_ROWS.append({
        "method": method,
        "avg_stability_tau": avg_stability,
        "avg_pruning_kept_ratio": avg_pruning_kept,
        "group_coverage": group_coverage.get(method, None),
        "num_configs_stability": len(stability_rows),
        "num_configs_pruning": len(pruning_mid)
    })

E5_POLICY_DF = pd.DataFrame(E5_POLICY_ROWS)
print("[E5] Policy summary (aggregated across all configurations):")
display(E5_POLICY_DF)
print("Experiment E5 completed.")


[E5] Policy summary (aggregated across all configurations):


,method,avg_stability_tau,avg_pruning_kept_ratio,group_coverage,num_configs_stability,num_configs_pruning
0,Shapley_MC,0.606566,0.225454,0.247376,36,36
1,Owen,0.621544,0.238824,0.244364,36,36
2,Banzhaf,0.575209,0.154692,NaN,36,36


Experiment E5 completed.


In [13]:
E5_OUT_DIR = os.path.join(OUT_ROOT, "experiments", "E5")
ensure_dir(E5_OUT_DIR)
E5_POLICY_DF.to_csv(os.path.join(E5_OUT_DIR, "E5_policy.csv"), index=False)
print(f"[E5] ✓ Zapisano wyniki do {E5_OUT_DIR}/")

[E5] ✓ Zapisano wyniki do ../Docs/Problems/shapley_values/experiments/E5/
